In [1]:
# Librerias necesarias
import os
import sys
import json
import time
import numpy as np
import pandas as pd
from copy import deepcopy
from generacion_pacientes import generar_pacientes
from collections import deque
import parametros as p
import kpis as kpi
from clases import *

In [94]:
# Cargar los datos
df = pd.read_csv("resultados simulacion/ModeloA_None_T4500_C4208/logs/0.csv")

In [13]:
# Salidas por ciclo de pacientes en ICU
# 1. Filtrar solo estancias en ICU y ordenar
df_icu = df[df["UNIDAD"] == "ICU"].copy()
df_icu = df_icu.sort_values(["ID", "TI"])

# 2. Obtener siguiente unidad y su tiempo para cada paciente
df_siguiente = df.sort_values(["ID", "TI"]).copy()
df_siguiente["unidad_siguiente"] = df_siguiente.groupby("ID")["UNIDAD"].shift(-1)
df_siguiente["ti_siguiente"] = df_siguiente.groupby("ID")["TI"].shift(-1)

# 3. Unir con ICU para ver su salida
df_icu = df_icu.merge(df_siguiente[["ID", "UNIDAD", "unidad_siguiente", "ti_siguiente"]], 
                      on=["ID", "UNIDAD"], how="left")

# 4. Detectar si salió de ICU (pasó a otra unidad o terminó en ICU)
df_icu_salidas = df_icu[
    df_icu["unidad_siguiente"].notna() | 
    (df_icu["TF"] == df_icu.groupby("ID")["TF"].transform("max"))
]

# 5. Determinar hora de salida
df_icu_salidas["hora_salida"] = df_icu_salidas["ti_siguiente"].fillna(df_icu_salidas["TF"])

# 6. Calcular ciclo (12 horas)
df_icu_salidas["ciclo"] = (df_icu_salidas["hora_salida"] // 12).astype(int)

# 7. Agrupar por ciclo
salidas_por_ciclo = df_icu_salidas.groupby("ciclo").size().reset_index(name="salidas_ICU")

# 8. Mostrar
display(salidas_por_ciclo[(salidas_por_ciclo["ciclo"] > 2000) & salidas_por_ciclo["ciclo"] < 3000].mean())

ciclo          2125.040970
salidas_ICU      39.887685
dtype: float64

In [12]:
# Porcentaje de evoluciones de pacientes que subieron de unidad
nivel_unidades = {"OR": 1, "ICU": 2, "SDU_WARD": 3}

# 2. Filtrar y mapear niveles
df_nivel = df[df["UNIDAD"].isin(nivel_unidades)].copy()
df_nivel["nivel"] = df_nivel["UNIDAD"].map(nivel_unidades)

# 3. Ordenar por tiempo por paciente
df_nivel = df_nivel.sort_values(["ID", "TI"])

# 4. Obtener lista de unidades por paciente
unidades_por_paciente = df_nivel.groupby("ID")["UNIDAD"].agg(list).reset_index()

# 5. Función para detectar subidas
def detectar_subidas(unidades):
    subidas = []
    niveles = [nivel_unidades[u] for u in unidades]
    for i in range(len(niveles) - 1):
        if niveles[i+1] < niveles[i]:  # subió de complejidad
            subidas.append((unidades[i], unidades[i+1]))
    return subidas

# 6. Aplicar y clasificar pacientes
unidades_por_paciente["subidas"] = unidades_por_paciente["UNIDAD"].apply(detectar_subidas)
unidades_por_paciente["subio"] = unidades_por_paciente["subidas"].apply(lambda x: len(x) > 0)

# 7. Porcentajes globales
total_pacientes = len(unidades_por_paciente)
pacientes_subieron = unidades_por_paciente["subio"].sum()
pacientes_no_subieron = total_pacientes - pacientes_subieron

porcentaje_subieron = round(100 * pacientes_subieron / total_pacientes, 2)
porcentaje_no_subieron = round(100 * pacientes_no_subieron / total_pacientes, 2)

print(f"📊 Pacientes que subieron al menos una vez: {porcentaje_subieron}%")
print(f"📉 Pacientes que nunca subieron: {porcentaje_no_subieron}%")

# 8. Proporciones de cada tipo de subida
from collections import Counter
todas_subidas = sum(unidades_por_paciente["subidas"], [])
conteo_subidas = Counter(todas_subidas)

df_subidas = pd.DataFrame(conteo_subidas.items(), columns=["De_A", "conteo"])
df_subidas[["de", "a"]] = pd.DataFrame(df_subidas["De_A"].tolist(), index=df_subidas.index)
df_subidas.drop(columns="De_A", inplace=True)
df_subidas["proporcion_%"] = (df_subidas["conteo"] / df_subidas["conteo"].sum() * 100).round(2)

display(df_subidas)

📊 Pacientes que subieron al menos una vez: 11.86%
📉 Pacientes que nunca subieron: 88.14%


,conteo,de,a,proporcion_%
0,9464,ICU,OR,49.53
1,2043,SDU_WARD,OR,10.69
2,7601,SDU_WARD,ICU,39.78


In [13]:
# Porcentaje de GRDs y requerimientos iniciales
# 1. Agrupar por ID y quedarte con MS_GRD y requerimiento_inicial únicos por paciente
df_tipos = df.groupby("ID").first()[["MS_GRD", "requerimiento_inicial"]].reset_index()

# 2. Contar ocurrencias de cada tipo de paciente
conteo_tipos = df_tipos.groupby(["MS_GRD", "requerimiento_inicial"]).size().reset_index(name="conteo")

# 3. Calcular el porcentaje
total_pacientes = conteo_tipos["conteo"].sum()
conteo_tipos["porcentaje"] = (conteo_tipos["conteo"] / total_pacientes * 100).round(2)
conteo_tipos
# conteo_tipos[(conteo_tipos["MS_GRD"] < 5) & (conteo_tipos["requerimiento_inicial"] == 1)].sort_values("porcentaje", ascending=False)

,MS_GRD,requerimiento_inicial,conteo,porcentaje
0,1,1,7245,5.19
1,1,2,7301,5.23
2,1,3,4234,3.03
3,2,1,6806,4.87
4,2,2,6803,4.87
5,2,3,3703,2.65
6,3,1,7812,5.59
7,3,2,7130,5.11
8,3,3,4292,3.07
9,4,1,6461,4.63


In [17]:
# Clase simulacion modificada
class SimulacionMod: # Revisado, funciona bien

    def __init__(self, T_max, seed, ciclos, modelo = Modelo(), modelo_alternativo = Modelo(), ciclo_de_cambio = 0, pacientes_caso_base = False, log_detallado = False):
        # Asi no vuelvo a crear el archivo de pacientes si ya existe
        """Se empieza generando los pacientes con la semilla de random
        y la cantidad de ciclos que se desean crear pacientes"""
        t0 = time.time()
        self.seed = seed
        self.pacientes_caso_base = pacientes_caso_base
        self.log_detallado = log_detallado
        if self.pacientes_caso_base == False:
            folder_path = "resultados incertidumbre"
            self.file_name = f"{self.seed}_{ciclos}.json"
            file_path = os.path.join(folder_path, self.file_name)
            if os.path.isfile(file_path):
                print(f"Se utiliza archivo existente {self.file_name} de pacientes ({time.time() - t0:.2f} segundos)")
            else:
                generar_pacientes(self.seed, ciclos)
                print(f"Se crea archivo {self.file_name} de pacientes ({time.time() - t0:.2f} segundos)")
        t0 = time.time()
        self.T_max = T_max
        self.pacientes_separados_por_llegada = self.cargar_pacientes_separados_por_llegada()
        print(f"Pacientes separados por llegada cargados ({time.time() - t0:.2f} segundos)")
        t0 = time.time()
        self.ciclos = ciclos
        self.modelo = modelo
        self.modelo_alternativo = modelo_alternativo
        self.ciclo_de_cambio = ciclo_de_cambio
        self.hospital_1 = Hospital(1)
        self.hospital_2 = Hospital(2)
        self.hospital_3 = Hospital(3)
        self.wl = WL([1, 2, 3], [5, 6, 7, 8])
        self.hospitales = { # Para acceder a los hospitales por su id
            0: self.wl,
            p.dict_hospitales["Hospital_1"]: self.hospital_1,
            p.dict_hospitales["Hospital_2"]: self.hospital_2,
            p.dict_hospitales["Hospital_3"]: self.hospital_3
        }
        self.ps = PS([1, 2, 3], [1, 2, 3, 4, 5, 6, 7, 8])
        self.end = END([1, 2, 3], [1, 2, 3, 4, 5, 6, 7, 8])
        self.unidades_termino = { # Para acceder a las unidades de termino por su id
            p.dict_unidades["PS"]: self.ps,
            p.dict_unidades["END"]: self.end
        }
        self.budget = p.budget
        self.tasa_descuento = p.tasa_descuento
        print(f"Clase Simulacion instanciada ({time.time() - t0:.2f} segundos)")

    # Funcion necesaria al momento de instanciar la clase
    def cargar_pacientes_separados_por_llegada(self): # Revisado, funciona bien
        # Cargo los datos necesarios y transformo las llaves nuevamente a int (que se habian vuelto str)
        incertidumbre = {}
        for hospital in range(0,4): # 0 son llegadas a WL
            incertidumbre[hospital] = {}
            for requerimiento in range(1,4):
                incertidumbre[hospital][requerimiento] = {}

        if self.pacientes_caso_base == False:
            with open(f"resultados incertidumbre/{self.file_name}", "r") as file:
                incertidumbre_keys_str = json.load(file)
        else:
            with open("resultados incertidumbre/incertidumbre_base.json", "r") as file:
                incertidumbre_keys_str = json.load(file)

        for hospital in range(0,4): # 0 son llegadas a WL
            for requerimiento in range(1,4):
                for grd in range(1,9):
                    incertidumbre[hospital][requerimiento][grd] = []
                    lista = incertidumbre_keys_str[str(hospital)][str(requerimiento)][str(grd)]
                    if lista != []:
                        for data in lista:
                            arreglado = {
                            'TI': data["TI"],
                            'camino': {1: data['camino']["1"], 2: data['camino']["2"], 3: data['camino']["3"]},
                            'espera': {1: data['espera']["1"], 2: data['espera']["2"], 3: data['espera']["3"]}
                            }
                            if self.pacientes_caso_base:
                                arreglado["decisiones"] = data["decisiones"]
                                arreglado["id"] = data["id"]

                            incertidumbre[hospital][requerimiento][grd].append(arreglado)
                    else:
                        incertidumbre[hospital][requerimiento][grd] = []

        # Se instancian a todos los pacientes a partir de los datos de incertidumbre
        pacientes = {}
        lista_pacientes = []
        # 0 son llegadas a WL
        for hospital in range(0,4): 
            pacientes[hospital] = {}
            for requerimiento in range(1,4):
                pacientes[hospital][requerimiento] = {}
                for grd in range(1,9):
                    pacientes[hospital][requerimiento][grd] = []
                    cantidad_pacientes = len(incertidumbre[hospital][requerimiento][grd])
                    if cantidad_pacientes != 0:
                        for i in range(cantidad_pacientes):
                            paciente = Paciente(hospital, requerimiento, grd, incertidumbre[hospital][requerimiento][grd][i]) # i es el index de la lista
                            pacientes[hospital][requerimiento][grd].append(paciente)
                            lista_pacientes.append(paciente)
                    else:
                        pacientes[hospital][requerimiento][grd] = []

        # Se generan tantas listas como dias con llegadas haya
        pacientes_separados_por_llegada = {}
        for paciente in lista_pacientes:
            ciclo = paciente.ti_inicial
            if ciclo not in pacientes_separados_por_llegada:
                pacientes_separados_por_llegada[ciclo] = []
            pacientes_separados_por_llegada[ciclo].append(paciente)

        return pacientes_separados_por_llegada
    
    # Funciones necesarias al momento de simular
    def agregar_pacientes_ciclo_a_wl(self, ciclo): # Revisado, funciona bien
        pacientes_ciclo = self.pacientes_separados_por_llegada.get(ciclo, [])
        for paciente in pacientes_ciclo:
            # hospital 0 es WL
            if paciente.hospital_llegada == 0:
                self.wl.agregar_paciente(paciente)
            else:
                pass

    def agregar_pacientes_ciclo_a_ed(self, ciclo): # Revisado, funciona bien
        pacientes_ciclo = self.pacientes_separados_por_llegada.get(ciclo, [])
        for paciente in pacientes_ciclo:
            # si hospital es 1, 2 o 3 llegan a ED
            if paciente.hospital_llegada != 0:
                self.hospitales[paciente.hospital_llegada].agregar_paciente(paciente, paciente.unidad_actual)
            else:
                pass

    def sacar_paciente(self, paciente): # Revisado, funciona bien
        # Retorna el paciente que se saca de la unidad, o None si no se pudo sacar
        if paciente.hospital_actual == 0:
            return self.hospitales[paciente.hospital_actual].sacar_paciente(paciente)  
        else:
            return self.hospitales[paciente.hospital_actual].sacar_paciente(paciente, paciente.unidad_actual)

    def agregar_paciente(self, paciente, hospital, unidad): # Revisado, funciona bien
        # Retorna True si se agrega el paciente, False si no
        if unidad not in (p.dict_unidades["PS"], p.dict_unidades["END"]):
            return self.hospitales[hospital].agregar_paciente(paciente, unidad)
        else:
            # No retornan nada porque siempre se agrega el paciente
            self.unidades_termino[unidad].agregar_paciente(paciente)
            return True

    def implementar_decisiones(self, decisiones: list): # Revisado, funciona bien
        """Las decisiones son una lista de diccionarios, cada uno con la siguiente estructura:
        {
            paciente: {"hospital": hospital, "unidad": unidad},
            ...
            paciente: {"hospital": hospital, "unidad": unidad},
        }
        Estas se deben implementar en el orden de la lista, se sacan a todos los pacientes de su
        unidad actual y se cambian a la unidad y hospital que se indica en el diccionario. Luego se 
        pasa al siguiente diccionario y se repite el proceso. Se hace de esta manera porque a veces
        puede ocurrir que dos pacientes tengan que ser cambiados entre si, por lo que los cambios
        deben ser simultaneos por cada diccionario, para sacar a un paciente no necesito saber donde
        esta ya que cada paciente contiene su unidad y hospital actual.
        """
        for decision in decisiones:
            # Saco a todos los pacientes de su unidad actual
            for paciente in decision:
                sacado = self.sacar_paciente(paciente)
                if paciente.esperando == False:
                    print(f"Error no esperando: id: {paciente.id}, h:{paciente.hospital_actual}, u: {paciente.unidad_actual}, caminos: {paciente.camino}, espera: {paciente.espera}")
                    
                if sacado == None:
                    print(f"Error al sacar paciente {paciente.id} de h:{paciente.hospital_actual}, u: {paciente.unidad_actual}")

            # Agrego a todos los pacientes a su nueva unidad
            for paciente, destino in decision.items():
                if self.agregar_paciente(paciente, destino["hospital"], destino["unidad"]):
                    pass
                else:
                    print(f"Error al agregar paciente {paciente.id} a h:{destino['hospital']}, u: {destino['unidad']}")
             
    def actualizar_tiempo(self): # Revisado, funciona bien
        self.hospital_1.actualizar_tiempo()
        self.hospital_2.actualizar_tiempo()
        self.hospital_3.actualizar_tiempo()
        self.wl.actualizar_tiempo()
        # Incrementar el tiempo del sistema
        self.T += 1

    def entregar_log_pacientes_terminados_como_data_frame(self): # Revisado, funciona bien
        t0 = time.time()
        log_completo = []
        for requerimiento in [1, 2, 3]:
            for grd in [1, 2, 3, 4, 5, 6, 7, 8]:
                for paciente in self.end.sub_listas[requerimiento][grd]:
                    contador = 0                    
                    for evento in paciente.log_eventos.copy():
                        contador += 1
                        evento["orden"] = contador
                        evento["requerimiento_inicial"] = paciente.requerimiento_inicial
                        log_completo.append(evento)
                    contador = 0

                for paciente in self.ps.sub_listas[requerimiento][grd]:
                    for evento in paciente.log_eventos.copy():
                        contador += 1
                        evento["orden"] = contador
                        evento["requerimiento_inicial"] = paciente.requerimiento_inicial
                        log_completo.append(evento)
                    
        # Convertir la lista de eventos a un DataFrame
        df_log = pd.DataFrame(log_completo)
        # Ordenar el DataFrame por ID y TI y TF
        df_log.sort_values(by=['ID', 'orden'], inplace=True)
        # Resetear el índice
        df_log.reset_index(drop=True, inplace=True)
        print(f"Log de pacientes terminado como DataFrame ({time.time() - t0:.2f} segundos)")
        return df_log

    def entregar_log_detallado_pacientes_terminados_como_data_frame(self): # En proceso
        t0 = time.time()

        def calcular_costo_espera(row):
            drg = row["MS_GRD"]
            los = row["LOS"]
            unidad = row.get("requerimiento_inicial", None)
            ubicacion = row["UBICACIÓN"]
            hospital_nombre = row["HOSPITAL"]
            hospital = p.dict_hospitales.get(hospital_nombre)

            # 1. Paciente en WL_WL (esperando en lista)
            if row["HOSPITAL"] == "WL" and ubicacion == "WL_WL":
                return p.dict_costo_espera_wl[drg][unidad] * los
                
            # 2. Paciente en GA o ED con LOS > 0
            elif row["UNIDAD"] in {"GA", "ED"} and los > 0:
                source = p.dict_costo_espera_ga if row["UNIDAD"] == "GA" else p.dict_costo_espera_ed
                return source[hospital][drg][unidad] * los
                
            # 3. Paciente hospitalizado y bloqueado
            elif row["UNIDAD"] in {"OR", "ICU", "SDU_WARD"} and "->" in ubicacion and los > 0:
                try:
                    origen_str, destino_str = ubicacion.split(" -> ")
                    unidad_actual = "_".join(origen_str.split("_")[2:])
                    unidad_requerida = "_".join(destino_str.split("_")[2:])
                    return p.dict_costo_espera_hospitalizado[hospital][drg][p.dict_unidades[unidad_actual]][p.dict_unidades[unidad_requerida]] * los
                except Exception:
                    return 0  # fallback in case of malformed UBICACION
            return 0

        def parse_hospital_number(hospital_str):
            return int(hospital_str.split("_")[1])

        log_completo = []
        for requerimiento in [1, 2, 3]:
            for grd in [1, 2, 3, 4, 5, 6, 7, 8]:
                pacientes = self.end.sub_listas[requerimiento][grd] + self.ps.sub_listas[requerimiento][grd]
                for paciente in pacientes:
                    contador = 0   
                    # for evento in paciente.log_eventos.copy():
                    tl = paciente.log_eventos.copy()
                    for i in range(len(tl) - 1):
                        evento = tl[i].copy()
                        contador += 1
                        evento["LOS"] = evento["TF"] - evento["TI"]
                        evento["COSTO DER WL"] = 0
                        evento["COSTO DER ED"] = 0
                        evento["COSTO TRASLADO"] = 0
                        evento["orden"] = contador
                        evento["requerimiento_inicial"] = paciente.requerimiento_inicial
                        evento["COSTO ESPERA"] = calcular_costo_espera(evento)
                        log_completo.append(evento)
                
                        row_current = evento.copy()
                        row_next = tl[i + 1]
                        time_gap = row_current['TF'] < row_next['TI']
                        time_gap_cero = row_current['TF'] == row_next['TI']
                        same_hospital = row_current['HOSPITAL'] == row_next['HOSPITAL']

                        new_row = {
                                'ID': row_current['ID'],
                                'MS_GRD': row_current['MS_GRD'],
                                'UBICACIÓN': f"{row_current['UBICACIÓN']} -> {row_next['UBICACIÓN']}",
                                'TI': row_current['TF'],
                                'TF': row_next['TI'],
                                'LOS': row_next['TI'] - row_current['TF'],
                                'HOSPITAL': row_current['HOSPITAL'],
                                'orden': row_current['orden'] + 0.1,  # Ajustar el orden
                                'requerimiento_inicial': row_current['requerimiento_inicial']
                        }
                        
                        if time_gap:
                            new_row.update({
                                'UNIDAD': row_current['UNIDAD']
                            })
                            
                            costo_espera = calcular_costo_espera(new_row)

                            new_row.update({
                                "COSTO DER WL": 0,
                                "COSTO DER ED": 0,
                                "COSTO TRASLADO": 0,
                                "COSTO ESPERA": costo_espera
                            })

                            log_completo.append(new_row)
                        
                        elif time_gap_cero and not same_hospital:
                            x = new_row.copy()
                            traslados = {
                                'Hospital_1_ED -> Hospital_2_ED',
                                'Hospital_1_ED -> Hospital_3_ED',
                                'Hospital_2_ED -> Hospital_1_ED',
                                'Hospital_2_ED -> Hospital_3_ED',
                                'Hospital_3_ED -> Hospital_1_ED',
                                'Hospital_3_ED -> Hospital_2_ED'
                            }
                            costo_der_wl = p.dict_costo_derivar_wl[x["MS_GRD"]][x["requerimiento_inicial"]] if x["UBICACIÓN"] == "WL_WL -> PS_PS" else 0
                            costo_der_ed = p.dict_costo_derivar_ed[p.dict_hospitales[x["HOSPITAL"]]][x["MS_GRD"]][x["requerimiento_inicial"]] if x["UBICACIÓN"] == f"{x['HOSPITAL']}_ED -> PS_PS" else 0
                            costo_traslado = (p.dict_costo_traslado[parse_hospital_number(x["UBICACIÓN"].split(" -> ")[0])]
                                              [parse_hospital_number(x["UBICACIÓN"].split(" -> ")[1])][x["MS_GRD"]][x["requerimiento_inicial"]]
                                            if x["UBICACIÓN"] in traslados else 0)

                            new_row.update({
                                'UNIDAD': "En movimiento",
                                "COSTO DER WL": costo_der_wl,
                                "COSTO DER ED": costo_der_ed,
                                "COSTO TRASLADO": costo_traslado,
                                "COSTO ESPERA": 0
                            })
                    
                            log_completo.append(new_row)
                    
                    evento = tl[-1].copy()
                    contador += 1
                    evento["LOS"] = evento["TF"] - evento["TI"]
                    evento["COSTO DER WL"] = 0
                    evento["COSTO DER ED"] = 0
                    evento["COSTO TRASLADO"] = 0
                    evento["COSTO ESPERA"] = 0
                    evento["orden"] = contador
                    evento["requerimiento_inicial"] = paciente.requerimiento_inicial
                    log_completo.append(evento)

        # Convertir la lista de eventos a un DataFrame
        df_log = pd.DataFrame(log_completo)
        # Ordenar el DataFrame por ID y TI y TF
        df_log.sort_values(by=['ID', 'orden'], inplace=True)
        df_log = df_log[['ID', 'MS_GRD', 'UBICACIÓN', 'TI', 'TF', 'LOS', 'HOSPITAL', 'UNIDAD', 'requerimiento_inicial', 'COSTO DER WL', 'COSTO DER ED', 'COSTO TRASLADO', 'COSTO ESPERA']]
        # Resetear el índice
        df_log.reset_index(drop=True, inplace=True)
        df_log[["TI", "TF", "LOS"]] = df_log[["TI", "TF", "LOS"]] * 12
        print(f"Log de pacientes terminado como DataFrame ({time.time() - t0:.2f} segundos)")
        return df_log

    def simular(self):
        t0 = time.time()
        self.T = 1  # Inicializa el tiempo del sistema en 1

        # Bucle principal de simulación
        while self.T <= self.T_max:
            if self.T % 1000 == 0:
                print(f"\nSimulando ciclo {self.T} de {self.T_max}")
            
            # Esto solo ocurre cuando quiero cambiar de modelo entremedio de la simulacion T!= 0
            if self.T == self.ciclo_de_cambio:
                self.modelo = self.modelo_alternativo
            
            # Agregar pacientes a la WL según el ciclo actual
            self.agregar_pacientes_ciclo_a_wl(self.T)

            # Agregar pacientes a las ED según el ciclo actual
            self.agregar_pacientes_ciclo_a_ed(self.T)

            # Entrego el estado de la simulación al modelo para tomar decisiones (argumento self)
            if self.T == 2593:
                return self.modelo.devolverme(self)

            decisiones = self.modelo.tomar_decisiones(self)

            # Se implementan todas las decisiones del modelo
            self.implementar_decisiones(decisiones)

            # Actualizar el tiempo de cada unidad y paciente
            self.actualizar_tiempo()
            
        # Al finalizar la simulación, se entregan los logs de los pacientes terminados
        print(f"Simulación finalizada en ({time.time() - t0:.2f} segundos)")
        if self.log_detallado:
            return self.entregar_log_detallado_pacientes_terminados_como_data_frame()
        else:
            return self.entregar_log_pacientes_terminados_como_data_frame()
        
    def __str__(self):
        pass



In [18]:
# Clase Modelo modificada (me entrega el estado actual para no tener que cargarlo manualmente)
class ModeloB(ModeloA):
    """Espera hasta que la Wl se llene con mas de 1000 personas, de ahi espera 100 ciclos mas
    y luego empieza a atender pacientes de la WL directamente, por lo que esta empieza a bajar"""
    def __init__(self):
        super().__init__()
    
    def devolverme(self, simulacion):
        # Reinicio las variables
        self.decisiones = []
        self.expulsados_wl_del_ciclo_actual = []
        self.actual = self.actual_vacio.copy()
        self.budget = p.budget

        # Copio localmente las ocupaciones de cada hospital
        self.cargar_ciclo(simulacion)
        # Reviso si hay pacientes que estoy obligado a sacar de WL (no implementado todavia)
        self.agregar_pacientes_obligatorio_a_ga() 
        # Primero reviso si hay pacientes que deben ser dados de alta
        self.dar_de_alta()

        return self.actual

In [19]:
# Corre la simulación y me entrega un estado a la mitad, la corta en el ciclo 2500
clase_modelo_base = ModeloB
T_max=4500
ciclos=4208
num_simulaciones=1
seed_inicial=0
clase_modelo_alternativo=None
ciclo_de_cambio=0
pacientes_caso_base=False
log_detallado=True

# Nombres base y alternativo para la carpeta principal
nombre_base = clase_modelo_base.__name__
nombre_alternativo = clase_modelo_alternativo.__name__ if clase_modelo_alternativo else "None"
nombre_carpeta = f"{nombre_base}_{nombre_alternativo}_T{T_max}_C{ciclos}"

# Crear carpeta principal y subcarpetas
base_dir = os.path.join("resultados simulacion", nombre_carpeta)
logs_dir = os.path.join(base_dir, "logs")
plots_dir = os.path.join(base_dir, "plots")
kpis_dir = os.path.join(base_dir, "kpis")
os.makedirs(logs_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(kpis_dir, exist_ok=True)

for i in range(num_simulaciones):
    seed = seed_inicial + i
    print(f"\n⏳ Simulación {i+1}/{num_simulaciones} con seed={seed}")

    # Reiniciar contador global
    Paciente.CONTADOR_ID = 1

    # Instanciar modelos
    modelo = clase_modelo_base()
    alternativo = clase_modelo_alternativo() if clase_modelo_alternativo else None

    # Ejecutar simulación
    simu = SimulacionMod(
        T_max, seed, ciclos,
        modelo=modelo,
        modelo_alternativo=alternativo,
        pacientes_caso_base=pacientes_caso_base,
        ciclo_de_cambio=ciclo_de_cambio,
        log_detallado=log_detallado
    ) 

    estado = simu.simular()


⏳ Simulación 1/1 con seed=0
Se utiliza archivo existente 0_4208.json de pacientes (0.00 segundos)
Pacientes separados por llegada cargados (3.56 segundos)
Clase Simulacion instanciada (0.00 segundos)

Simulando ciclo 1000 de 4500

Simulando ciclo 2000 de 4500


In [20]:
actual = deepcopy(estado)

# def pretty_print_dict(d, indent=0):
#     for key, value in d.items():
#         prefix = " " * indent + f"{key}: "
#         if isinstance(value, dict):
#             print(prefix)
#             pretty_print_dict(value, indent + 4)
#         elif isinstance(value, list):
#             print(prefix + f"list of length {len(value)}")
#         elif isinstance(value, deque):
#             print(prefix + f"deque of length {len(value)}")
#         else:
#             print(prefix + str(value))

# pretty_print_dict(actual)

# hoy = 0
# antes = 0
# for req in range(1, 4):
#     for grd in range(5, 9):
#         for paciente in actual["WL_sub_deques"][req][grd]:
#             if paciente.ti_evento_actual == 2500:
#                 hoy += 1
#             else:
#                 antes += 1

# print(f"Pacientes en WL hoy: {hoy}, antes: {antes}, total: {hoy + antes}")

In [ ]:
def correr_multiples_simulaciones(
    clase_modelo_base,
    T_max=4500,
    ciclos=4208,
    num_simulaciones=5,
    seed_inicial=0,
    clase_modelo_alternativo=None,
    ciclo_de_cambio=0,
    pacientes_caso_base=False,
    log_detallado=True
):
    # Nombres base y alternativo para la carpeta principal
    nombre_base = clase_modelo_base.__name__
    nombre_alternativo = clase_modelo_alternativo.__name__ if clase_modelo_alternativo else "None"
    nombre_carpeta = f"{nombre_base}_{nombre_alternativo}_T{T_max}_C{ciclos}"
    
    # Crear carpeta principal y subcarpetas
    base_dir = os.path.join("resultados simulacion", nombre_carpeta)
    logs_dir = os.path.join(base_dir, "logs")
    plots_dir = os.path.join(base_dir, "plots")
    kpis_dir = os.path.join(base_dir, "kpis")
    os.makedirs(logs_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)
    os.makedirs(kpis_dir, exist_ok=True)

    for i in range(num_simulaciones):
        seed = seed_inicial + i
        print(f"\n⏳ Simulación {i+1}/{num_simulaciones} con seed={seed}")

        # Reiniciar contador global
        Paciente.CONTADOR_ID = 1

        # Instanciar modelos
        modelo = clase_modelo_base()
        alternativo = clase_modelo_alternativo() if clase_modelo_alternativo else None

        # Ejecutar simulación
        simu = Simulacion(
            T_max, seed, ciclos,
            modelo=modelo,
            modelo_alternativo=alternativo,
            pacientes_caso_base=pacientes_caso_base,
            ciclo_de_cambio=ciclo_de_cambio,
            log_detallado=log_detallado
        )
        df = simu.simular()

        # Calcular KPIs
        t0 = time.time()
        kpis = kpi.calcular_kpis(
            df,
            save_plot=True,
            modelo=modelo,
            seed=seed,
            ciclos=ciclos,
            save_dir=plots_dir
        )
        print(f"✅ KPIs calculados en {time.time() - t0:.2f} segundos")

        # Guardar log como CSV
        filename_prefix = f"{seed}"
        ruta_csv = os.path.join(logs_dir, f"{filename_prefix}.csv")
        df.to_csv(ruta_csv, index=False)
        print(f"📄 Log guardado en: {ruta_csv}")

        # Guardar KPIs como JSON
        ruta_json = os.path.join(kpis_dir, f"{filename_prefix}.json")
        with open(ruta_json, 'w') as f:
            json.dump(kpis, f, indent=4)
        print(f"📊 KPIs guardados en: {ruta_json}")

correr_multiples_simulaciones(
    clase_modelo_base=ModeloProactivo,
    T_max=4500,
    ciclos=4208,
    num_simulaciones=10,
    seed_inicial=1,
    clase_modelo_alternativo=None,
    pacientes_caso_base=False,
    ciclo_de_cambio=0
)


In [141]:
def valor_esperado_pmf(dict_los):
    dict_respuesta = {}
    for grd in dict_los:
        pmf = dict_los[grd]["final_kde_pmf"]
        dict_respuesta[grd] = sum((i + 1) * p for i, p in enumerate(pmf))
    return dict_respuesta

los_or_wl = {
    "5": {
            "final_kde_pmf": [
                0.9912493923189111,
                0.008750607681088965
            ]
        },
        "6": {
            "final_kde_pmf": [
                1.0,
                0
            ]
        },
        "7": {
            "final_kde_pmf": [
                0.9036096786989289,
                0.096390321301071
            ]
        },
        "8": {
            "final_kde_pmf": [
                0.9171270718232044,
                0.08287292817679558
            ]
        }
}

los_icu_wl = {
            "5": {
                "final_kde_pmf": [
                    0.0998596092317132,
                    0.2399392151543154,
                    0.20238382209678374,
                    0.13779861874437685,
                    0.09546999948792478,
                    0.06380727367056496,
                    0.044603156510969806,
                    0.043296038059611464,
                    0.016576593042212446,
                    0.014203957142765991,
                    0.009966218328020875,
                    0.006529186706348545,
                    0.0038665904665149582,
                    0.005354117296507762,
                    0.004299643611361083,
                    0.0019854182085270932,
                    0.0016274057659139611,
                    0.0011163784817626561,
                    0.001272607191838671,
                    0.001360503368223033,
                    0.001025268436840452,
                    0.0005565823066124069,
                    0.00020178373253711696,
                    0.000423131107766943,
                    0.00015622871007601498,
                    0.00017900622130656598,
                    0.000289679908921479,
                    0.00015622871007601498,
                    0.00015622871007601498,
                    0.00015622871007601498,
                    0.00015622871007601498,
                    0.00017900622130656598,
                    0.000289679908921479,
                    0.00015622871007601498,
                    0.00015622871007601498,
                    0.00015622871007601498,
                    0.00015622871007601498,
                    0.000133451198845464
                ]
            },
            "6": {
                "final_kde_pmf": [
                    0.1914962465727857,
                    0.3371427159469176,
                    0.21056642229460348,
                    0.11871041563446809,
                    0.058210904514619494,
                    0.032063585149711235,
                    0.018254029852287766,
                    0.015371868289531914,
                    0.005934370137236921,
                    0.005517460099004955,
                    0.0029320760150119856,
                    0.000989061689539486,
                    0.0008848341799814955,
                    0.0003296872298464953,
                    0.0003296872298464953,
                    0.0003818009846254906,
                    0.0006072607049139954,
                    0.0002775734750675001
                ]
            },
            "7": {
                "final_kde_pmf": [
                    0.06650793169683784,
                    0.13234009307035205,
                    0.13342448112801963,
                    0.11526664651919992,
                    0.09803196801256718,
                    0.07788928641872823,
                    0.06821894667503887,
                    0.0587223461932434,
                    0.037577511296707534,
                    0.0290226064210069,
                    0.027608563535895,
                    0.02166768860625225,
                    0.01734100775795801,
                    0.015562105564148027,
                    0.015946424493224805,
                    0.008633902076209789,
                    0.008188886848008772,
                    0.008020702578565048,
                    0.006591834949050789,
                    0.006015237254175265,
                    0.004196257100613815,
                    0.004159655297844599,
                    0.004530308201513098,
                    0.003639119026116982,
                    0.003138042374768057,
                    0.002766230752105474,
                    0.002526001596117399,
                    0.0031929450789218825,
                    0.0019128020984726577,
                    0.0024882410743540992,
                    0.001986005704011091,
                    0.0008545873726268996,
                    0.0013179035022125246,
                    0.0009289496971594161,
                    0.0016336537017271995,
                    0.0008911891753961164,
                    0.0006498013004139581,
                    0.0006315003990293497,
                    0.0007424645263310832,
                    0.00046447484857970806,
                    0.0007424645263310832,
                    0.00024138787498215826,
                    0.00039011252404719146,
                    0.0005754389758814416,
                    0.00024138787498215826,
                    0.00031575019951467486,
                    0.00040841342543179986,
                    0.00024138787498215826,
                    0.00024138787498215826,
                    0.00024138787498215826,
                    0.00024138787498215826,
                    0.00024138787498215826,
                    0.00024138787498215826,
                    0.00024138787498215826,
                    0.00016702555044964163
                ]
            },
            "8": {
                "final_kde_pmf": [
                    0.0659090909090909,
                    0.13909090909090874,
                    0.15212121212121027,
                    0.12818181818181815,
                    0.09878787878787985,
                    0.07787878787878878,
                    0.06212121212121214,
                    0.05954545454545462,
                    0.04151515151515141,
                    0.036060606060606036,
                    0.026363636363636478,
                    0.019090909090909172,
                    0.012424242424242437,
                    0.013030303030303048,
                    0.010151515151515167,
                    0.007424242424242445,
                    0.007878787878787897,
                    0.0068181818181818395,
                    0.0066666666666666844,
                    0.003939393939393949,
                    0.0025757575757575815,
                    0.002727272727272733,
                    0.002878787878787885,
                    0.0015151515151515193,
                    0.001515151515151519,
                    0.0009090909090909113,
                    0.0009090909090909113,
                    0.001060606060606063,
                    0.001969696969696975,
                    0.0025757575757575815,
                    0.0010606060606060633,
                    0.0007575757575757594,
                    0.0010606060606060633,
                    0.0007575757575757594,
                    0.0010606060606060633,
                    0.0004545454545454557,
                    0.0004545454545454557,
                    0.0004545454545454557,
                    0.0003030303030303038
                ]}
}

los_sdu_wl = {
    "5": {
                "final_kde_pmf": [
                    0.0015729768682477823,
                    0.007791344832762731,
                    0.021824572209389635,
                    0.03892248114390559,
                    0.058833285003642696,
                    0.08717835330830104,
                    0.1119258980962447,
                    0.0746373054747533,
                    0.08017803595376372,
                    0.07153798518575746,
                    0.06440510444765299,
                    0.05747761432143556,
                    0.05355254868086806,
                    0.054389893037635,
                    0.03375434339145597,
                    0.026884848291214826,
                    0.023909329866541318,
                    0.019190161835079128,
                    0.018144114527037177,
                    0.014490483763421541,
                    0.013467555323530504,
                    0.01373891105824279,
                    0.008124720212995646,
                    0.007183301378429805,
                    0.003824066075437735,
                    0.004602861410551279,
                    0.0038124670702424923,
                    0.004017778539889943,
                    0.0046997572138972,
                    0.0019217384465013857,
                    0.001592387685812817,
                    0.0021890695616693955,
                    0.0017318923171142587,
                    0.0010577254554767976,
                    0.0010964679483672392,
                    0.0011313441061925995,
                    0.0008291368331992288,
                    0.0005269295602058579,
                    0.0004184347516646955,
                    0.0007555181824834268,
                    0.0006392907038121016,
                    0.0003022072729933706,
                    0.000263464780102929,
                    0.0001511036364966853,
                    0.0001511036364966853,
                    0.0001511036364966853,
                    0.0001511036364966853,
                    0.0001511036364966853,
                    0.0001511036364966853,
                    0.0001511036364966853,
                    0.0001511036364966853,
                    0.0001511036364966853,
                    0.00011236114360624367
                ]
            },
            "6": {
                "final_kde_pmf": [
                    0.00161807092646747,
                    0.015155332921949165,
                    0.05314510232906434,
                    0.09428071816733012,
                    0.12151165098192185,
                    0.11926065943191366,
                    0.1264570505175449,
                    0.12374117317868377,
                    0.077798739939691,
                    0.06059893233673072,
                    0.0487164019697687,
                    0.037543678968198484,
                    0.03017475771231255,
                    0.022910431892580005,
                    0.018041919098808003,
                    0.01117272300157024,
                    0.0065402184399711225,
                    0.005592620173533273,
                    0.0061450887174764775,
                    0.006277392847932443,
                    0.002592486552771165,
                    0.0017369663769905151,
                    0.0018817878529427772,
                    0.0014607321050189139,
                    0.0009869329717999893,
                    0.0008027767904855881,
                    0.000328977657266663,
                    0.000328977657266663,
                    0.0004210557479238635,
                    0.0005658772238761254,
                    0.000328977657266663,
                    0.000328977657266663,
                    0.000328977657266663,
                    0.000328977657266663,
                    0.000328977657266663,
                    0.000328977657266663,
                    0.0002368995666094625
                ]
            },
            "7": {
                "final_kde_pmf": [
                    0.00016106373396104155,
                    0.0009213745238864383,
                    0.004285596298809771,
                    0.008607709589903147,
                    0.016328008206304506,
                    0.03156849185636453,
                    0.03340218258500821,
                    0.037277621666502024,
                    0.03758955595222877,
                    0.045303349691399805,
                    0.04612735243099657,
                    0.05437272925572939,
                    0.0750950541757945,
                    0.04808672476438381,
                    0.04618056748270605,
                    0.044677778825347195,
                    0.04089235550361294,
                    0.04024810056776877,
                    0.037949051559733965,
                    0.03586257904277058,
                    0.03841526098642465,
                    0.02577953946866273,
                    0.022557981073675175,
                    0.021430534935947824,
                    0.01981112299297465,
                    0.017672286571601293,
                    0.020105224356209227,
                    0.022577232574999963,
                    0.012160170056175337,
                    0.013475571169484937,
                    0.012384642024041308,
                    0.007909955886583006,
                    0.008464762494200657,
                    0.008831614126235997,
                    0.009314805328119123,
                    0.00823265078603821,
                    0.004688681207362204,
                    0.005377660307319626,
                    0.005556840679539208,
                    0.0043756120585693925,
                    0.0036688000361199736,
                    0.003498394267263105,
                    0.004536959508296984,
                    0.0023174493620598314,
                    0.002586078062505937,
                    0.002084770222364265,
                    0.0024519055701661616,
                    0.0011543373793485565,
                    0.0008947669980317252,
                    0.0009932736453875152,
                    0.0005995307717309077,
                    0.0005726395301096421,
                    0.0005816978492389147
                ]
            },
            "8": {
                "final_kde_pmf": [
                    0.005696707303178761,
                    0.013786031673692583,
                    0.02871140480802104,
                    0.03828187307736133,
                    0.040674490144696415,
                    0.044377349891762605,
                    0.046940868178193054,
                    0.05389085108807106,
                    0.057593710835137105,
                    0.05839124985758213,
                    0.05605559986327886,
                    0.049504386464623504,
                    0.04819414378489235,
                    0.04357981086931759,
                    0.0438646462344765,
                    0.03907941209980635,
                    0.034977782841517625,
                    0.032756066993277946,
                    0.03332573772359579,
                    0.02711632676313095,
                    0.022274125555428997,
                    0.01845733166229921,
                    0.018628232881394577,
                    0.01657741825225021,
                    0.015438076791614441,
                    0.013786031673692597,
                    0.010538908510880692,
                    0.010424974364817116,
                    0.00791842315141847,
                    0.005525806084083398,
                    0.006152443887433062,
                    0.006038509741369487,
                    0.005240970718924459,
                    0.005468839011051611,
                    0.00569670730317876,
                    0.004956135353765522,
                    0.0045003987695112204,
                    0.0025635182864304417,
                    0.0020508146291443544,
                    0.003190156089780106,
                    0.0035319585279708315,
                    0.003987695112225132,
                    0.0024495841403668664,
                    0.00039876951122251326,
                    0.0010823743876039646,
                    0.0021647487752079293,
                    0.002050814629144354,
                    0.0012532756066993274,
                    0.0006266378033496637,
                    0.0002278682921271504
                ]
            }
        }

# final_kde_pmf

esperado = {
    5: {
        1: 1.69,
        2: 1.87,
        3: 1.73
    },
    6: {
        1: 1.36,
        2: 1.48,
        3: 1.26
    },
    7: {
        1: 1.52,
        2: 1.55,
        3: 1.44
    },
    8: {
        1: 0.89,
        2: 0.86,
        3: 0.78
    }
}

costos = {5: {
        1: 958.0795336,
        2: 837.5737947,
        3: 515.5510516
    },
    6: {
        1: 650.8546582,
        2: 540.5882262,
        3: 310.7158418
    },
    7: {
        1: 1561.945032,
        2: 1411.735342,
        3: 836.2311842
    },
    8: {
        1: 1355.643898,
        2: 1251.685351,
        3: 758.4039779
    }}

listas = [los_or_wl, los_icu_wl, los_sdu_wl]
los_esperado = {5: {}, 6: {}, 7: {}, 8: {}}
for i in range(1, 4):
    for grd, valor in valor_esperado_pmf(listas[i-1]).items():
        los_esperado[int(grd)][i] = valor
        
display(los_esperado)


costos_mod ={}
for grd in costos:
    costos_mod[grd] = {}
    for unidad in costos[grd]:
        costos[grd][unidad] = round(costos[grd][unidad], 2)
        costos_mod[grd][unidad] = costos[grd][unidad] - costos[grd][unidad + 1] if unidad < 3 else costos[grd][unidad]
        tiempo_esperado = los_esperado[grd][unidad]
        costos_mod[grd][unidad] = costos_mod[grd][unidad]/tiempo_esperado

display(costos_mod)

for grd in costos_mod:
    costo_paciente_or = sum(costos_mod[grd].values())
    print(f"GRD {grd}: Costo por paciente = {costo_paciente_or:.2f}")

re_mod = {}
for grd in costos_mod:
    for unidad in costos[grd]:
        if unidad == 1:
            re_mod[grd, unidad] = round(sum(costos_mod[grd].values()),2)
        elif unidad == 2:
            re_mod[grd, unidad] = round(costos_mod[grd][unidad] + costos_mod[grd][unidad - 1],2)
        elif unidad == 3:
            re_mod[grd, unidad] = round(costos_mod[grd][unidad],2)

display(re_mod)


{5: {1: 1.008750607681089, 2: 4.182641653293938, 3: 11.13118679685788},
 6: {1: 1.0, 2: 2.901709649600794, 3: 8.014596277066499},
 7: {1: 1.096390321301071, 2: 7.019947062392764, 3: 17.782867345274333},
 8: {1: 1.0828729281767955, 2: 6.200151515151526, 3: 14.485074626865677}}

{5: {1: 119.46085026607231, 2: 76.98937061615158, 3: 46.31581604088522},
 6: {1: 110.26177380000001, 2: 79.22024804639749, 3: 38.769264134877886},
 7: {1: 137.00837656222873, 2: 81.98193101527984, 3: 47.02447494904257},
 8: {1: 95.99893606632655, 2: 79.56031733975207, 3: 52.35734157650693}}

GRD 5: Costo por paciente = 242.77
GRD 6: Costo por paciente = 228.25
GRD 7: Costo por paciente = 266.01
GRD 8: Costo por paciente = 227.92


{(5, 1): 242.77,
 (5, 2): 196.45,
 (5, 3): 46.32,
 (6, 1): 228.25,
 (6, 2): 189.48,
 (6, 3): 38.77,
 (7, 1): 266.01,
 (7, 2): 218.99,
 (7, 3): 47.02,
 (8, 1): 227.92,
 (8, 2): 175.56,
 (8, 3): 52.36}

In [2]:
modelo = ModeloProactivo()

In [4]:
modelo.demanda_ed

for hospital in [1,2,3]:
    for requerimiento in [1,2,3]:
        demanda = modelo.demanda_ed[hospital][requerimiento]["total_esperado"]
        print(f"Hospital {hospital}, Requerimiento {requerimiento}: Demanda esperada = {demanda:.2f}")
        

Hospital 1, Requerimiento 1: Demanda esperada = 2.40
Hospital 1, Requerimiento 2: Demanda esperada = 2.66
Hospital 1, Requerimiento 3: Demanda esperada = 1.60
Hospital 2, Requerimiento 1: Demanda esperada = 2.32
Hospital 2, Requerimiento 2: Demanda esperada = 2.26
Hospital 2, Requerimiento 3: Demanda esperada = 1.15
Hospital 3, Requerimiento 1: Demanda esperada = 1.98
Hospital 3, Requerimiento 2: Demanda esperada = 1.63
Hospital 3, Requerimiento 3: Demanda esperada = 0.78
